# Word2Vec：词向量的革命
> 难度标签：高级 | 预计时长：10-20分钟 | 前置知识：无学习经验


> 所属模块：08_自然语言处理 | 源文件：Word2Vec.py | 核心功能：Skip-gram/CBOW、词向量训练、语义相似度

## 概述

Word2Vec（2013）将每个词映射到一个低维稠密向量，语义相似的词在向量空间中距离近。经典发现：king - man + woman ≈ queen。

**导入必要的库**

In [ ]:
# 确保中文输出正常（Windows 环境兼容）
import sys
try:
    sys.stdout.reconfigure(encoding='utf-8')
except AttributeError:
    pass

import numpy as np

try:
    from gensim.models import Word2Vec, KeyedVectors
    from gensim.utils import simple_preprocess
    HAS_GENSIM = True
except ImportError:
# --- 赋值/配置 ---
    HAS_GENSIM = False
    print("[SKIP] gensim 未安装，跳过本示例")
    import sys; sys.exit(0)
    HAS_GENSIM = False
    print("gensim 未安装,请运行: pip install gensim\n")


## 数学原理

### 1. Word2Vec 的核心思想

Word2Vec 将每个词映射为一个 $d$ 维稠密向量（词嵌入），使得**语义相似的词在向量空间中距离相近**。

设词汇表为 $V$，词 $w$ 的嵌入向量为 $\mathbf{v}_w \in \mathbb{R}^d$，嵌入矩阵 $W \in \mathbb{R}^{|V| \times d}$。

### 2. CBOW 模型（Continuous Bag of Words）

CBOW 用上下文词预测中心词。

**输入**：中心词 $w_t$ 的上下文窗口 $\{w_{t-c}, \ldots, w_{t-1}, w_{t+1}, \ldots, w_{t+c}\}$

**上下文向量**（平均）：

$$\mathbf{h} = \frac{1}{2c}\sum_{-c \leq j \leq c, j \neq 0} \mathbf{v}_{w_{t+j}}$$

**输出**：通过 softmax 计算中心词的概率：

$$P(w_t | \text{context}) = \frac{\exp(\mathbf{u}_{w_t}^\top \mathbf{h})}{\sum_{w \in V} \exp(\mathbf{u}_w^\top \mathbf{h})}$$

其中 $\mathbf{u}_w$ 是词 $w$ 的输出嵌入向量。

### 3. Skip-gram 模型

Skip-gram 用中心词预测上下文词（代码中 `sg=1` 使用此模型）。

**目标**：最大化给定中心词 $w_t$ 时上下文词 $w_{t+j}$ 的对数似然：

$$\mathcal{L} = \sum_{t=1}^{T}\sum_{-c \leq j \leq c, j \neq 0} \log P(w_{t+j} | w_t)$$

其中：

$$P(w_o | w_i) = \frac{\exp(\mathbf{u}_{w_o}^\top \mathbf{v}_{w_i})}{\sum_{w \in V} \exp(\mathbf{u}_w^\top \mathbf{v}_{w_i})}$$

### 4. 负采样（Negative Sampling）

直接计算 softmax 的分母需遍历整个词汇表 $|V|$，计算量巨大。负采样将其简化为二分类问题：

$$\mathcal{L}_{neg} = \log \sigma(\mathbf{u}_{w_o}^\top \mathbf{v}_{w_i}) + \sum_{k=1}^{K} \mathbb{E}_{w_k \sim P_n(w)}[\log \sigma(-\mathbf{u}_{w_k}^\top \mathbf{v}_{w_i})]$$

其中 $\sigma$ 是 sigmoid 函数，$K$ 是负样本数，$P_n(w) \propto f(w)^{3/4}$ 是噪声分布（词频的 3/4 次幂）。

### 5. 词向量的线性类比关系

训练好的词向量满足著名的类比关系：

$$\mathbf{v}_{\text{king}} - \mathbf{v}_{\text{man}} + \mathbf{v}_{\text{woman}} \approx \mathbf{v}_{\text{queen}}$$

这表明词向量空间中存在**线性子结构**，能捕捉语义和语法关系。

### 6. 余弦相似度

词向量间的相似度用余弦相似度衡量：

$$\text{sim}(w_i, w_j) = \cos(\theta) = \frac{\mathbf{v}_{w_i} \cdot \mathbf{v}_{w_j}}{\|\mathbf{v}_{w_i}\| \cdot \|\mathbf{v}_{w_j}\|}$$

代码中 `model.wv.most_similar("学习")` 返回余弦相似度最高的词。

### 7. 代码与数学的对应

| 代码 | 数学含义 |
|------|----------|
| `vector_size=50` | 嵌入维度 $d=50$ |
| `window=3` | 上下文窗口 $c=3$ |
| `sg=1` | 使用 Skip-gram 模型 |
| `sg=0` | 使用 CBOW 模型 |
| `min_count=1` | 保留所有词（无低频词过滤） |
| `epochs=100` | 训练轮数 |
| `most_similar()` | $\arg\max_{w'} \cos(\mathbf{v}_w, \mathbf{v}_{w'})$ |
| `similarity(w1, w2)` | $\cos(\mathbf{v}_{w_1}, \mathbf{v}_{w_2})$ |

### 1. 训练语料

执行模型训练循环，观察损失函数的收敛过程。

In [ ]:
sentences = [
    list("机器学习是人工智能的基础"),
    list("深度学习是机器学习的子领域"),
    list("自然语言处理用到深度学习"),
    list("计算机视觉依赖卷积神经网络"),
# --- 继续 ---
    list("强化学习通过奖励训练智能体"),
    list("Python是数据科学的首选语言"),
    list("神经网络是深度学习的核心模型"),
    list("特征工程对传统机器学习很重要"),
    list("过拟合是机器学习的常见问题"),
# --- 继续 ---
    list("交叉验证用于评估模型泛化能力"),
]

print("=== 训练语料 ===")
for i, s in enumerate(sentences):
    print(f"  文档{i+1}: {''.join(s)}")

### 2. 训练 Word2Vec

执行模型训练循环，观察损失函数的收敛过程。

In [ ]:
# vector_size: 词向量维度
# window: 上下文窗口大小
# min_count: 忽略出现次数低于此值的词
# sg: 0=CBOW, 1=Skip-gram
model = Word2Vec(
    sentences=sentences,
    vector_size=50,
    window=3,
    min_count=1,
# --- 赋值/配置 ---
    sg=1,  # Skip-gram
    epochs=100,
    workers=1,
    seed=42,
)

print(f"\n=== Word2Vec 模型 ===")
print(f"词汇表大小: {len(model.wv)}")
print(f"词向量维度: {model.wv.vector_size}")

### 3. 词向量查询

运行 3. 词向量查询 的代码，观察算法在该环节的行为。

In [ ]:
word = "学习"
if word in model.wv:
    vec = model.wv[word]
    print(f"\n'{word}' 的词向量 (前10维): {vec[:10].round(4)}")
    print(f"向量范数: {np.linalg.norm(vec):.4f}")

### 4. 相似词查询

运行 4. 相似词查询 的代码，观察算法在该环节的行为。

In [ ]:
print("\n=== 最相似的词 ===")
for word in ["学习", "神经", "深度"]:
    if word in model.wv:
        similar = model.wv.most_similar(word, topn=3)
        print(f"  '{word}' 最相似: {[(w, f'{s:.3f}') for w, s in similar]}")

### 5. 词向量运算

运行 5. 词向量运算 的代码，观察算法在该环节的行为。

In [ ]:
print("\n=== 词向量运算（类比推理）===")
# king - man + woman ≈ queen
# 简单示例: 深度 - 机器 + 特征 ≈ ?
try:
    result = model.wv.most_similar(
        positive=["深度", "特征"],
        negative=["机器"],
        topn=3
# --- 继续 ---
    )
    print(f"  '深度' - '机器' + '特征' ≈ {[(w, f'{s:.3f}') for w, s in result]}")
except KeyError:
    print("  部分词不在词汇表中")

### 6. 两种训练模式

执行模型训练循环，观察损失函数的收敛过程。

In [ ]:
print("\n=== CBOW vs Skip-gram ===")
print("CBOW (sg=0): 用上下文预测中心词,速度快,适合高频词")
print("Skip-gram (sg=1): 用中心词预测上下文,适合低频词,语义更丰富")

# 训练 CBOW 对比
model_cbow = Word2Vec(sentences, vector_size=50, window=3, sg=0, epochs=100, workers=1, seed=42)
model_sg = Word2Vec(sentences, vector_size=50, window=3, sg=1, epochs=100, workers=1, seed=42)

if "学习" in model_cbow.wv and "学习" in model_sg.wv:
    print(f"\nCBOW '学习'向量前5维: {model_cbow.wv['学习'][:5].round(4)}")
    print(f"Skip-gram '学习'向量前5维: {model_sg.wv['学习'][:5].round(4)}")

### 7. 参数影响

探索关键超参数对模型行为的影响。

In [ ]:
print("\n=== 参数影响 ===")
for vs in [10, 50, 100]:
    m = Word2Vec(sentences, vector_size=vs, window=3, min_count=1, sg=1, epochs=100, workers=1, seed=42)
    print(f"  vector_size={vs:>3}: 词汇={len(m.wv)}, 向量维度={m.wv.vector_size}")

for win in [2, 3, 5]:
    m = Word2Vec(sentences, vector_size=50, window=win, min_count=1, sg=1, epochs=100, workers=1, seed=42)
    if "学习" in m.wv and "深度" in m.wv:
        sim = m.wv.similarity("学习", "深度")
        print(f"  window={win}: '学习'-'深度' 相似度={sim:.4f}")

print("\n=== Word2Vec 要点 ===")
print("- 分布式假设: 上下文相似的词,语义也相似")
print("- 词向量可以捕捉语义关系（类比推理）")
print("- vector_size 通常 100~300（越大表达力越强但需更多数据）")
print("- 预训练词向量: Google Word2Vec / GloVe / FastText")
# --- 输出结果 ---
print("- 局限: 每个词只有一个向量（多义词问题 → BERT）")

## 关键代码解释

```python
from gensim.models import Word2Vec
model = Word2Vec(sentences, vector_size=100, window=5, min_count=1, sg=1)
# sg=1: Skip-gram, sg=0: CBOW
model.wv.most_similar("机器学习", topn=5)
```

## 注意事项

1. 需要大量语料训练
2. 一词多义问题无法解决（同一个词只有一个向量）
3. OOV（词表外）词没有向量

## 总结与延伸

以上代码展示了 **Word2Vec** 的完整流程。

**解读要点**：
- 观察 **词汇表大小** 和 **向量维度** 是否合理
- 检查相似词查询结果是否符合语义直觉
- 关注分类任务中的 **TF-IDF 权重** 分布

**进一步思考**：尝试修改关键参数（如正则化强度、学习率、树深度等），观察结果如何变化。

### 延伸阅读

- **FastText**：子词级别的词向量，解决 OOV
- **GloVe**：基于共现矩阵的词向量
- **上下文词向量**：ELMo/BERT 每个词的向量随上下文变化
